# NowcastMY v4 -- Autonomous Macroeconomic Nowcasting

An LLM agent autonomously improves a GDP nowcasting model for Malaysia.

**Improvements over v3:**
- Resume-safe: state recovered from TSV on restart
- Multi-key API rotation (comma-separate keys)
- Best-commit git tagging (`best` tag) for safe revert
- Structured history summary in prompt (kept/failed, not raw TSV)
- Config block extracted separately to reduce prompt token waste
- Phase-based search: features -> model -> free
- Temperature annealing + failure-escape boost
- Coverage target surfaced in prompt alongside RMSE
- `COVID_WEIGHT` float replaces binary `EXCLUDE_COVID`

---
## Cell 0: Configuration

In [ ]:
#@title Configuration { display-mode: "form" }
# Comma-separate multiple keys for rotation: "key1,key2,key3"
OPENROUTER_API_KEYS = "sk-or-v1-xxxx"  #@param {type:"string"}
#@markdown Verified free models: `nvidia/nemotron-3-super-120b-a12b:free`, `meta-llama/llama-3.3-70b-instruct:free`, `qwen/qwen3-235b-a22b:free`
LLM_MODEL = "openrouter/free"  #@param {type:"string"}
RUN_TAG = "run1"  #@param {type:"string"}
MAX_EXPERIMENTS = 100  #@param {type:"integer"}
MAX_CONSECUTIVE_FAILURES = 15  #@param {type:"integer"}
DATA_DIR = "./data"  #@param {type:"string"}

# Phase boundaries
PHASE1_END = 30   # experiments 1-30:  FEATURE_COLS only
PHASE2_END = 60   # experiments 31-60: MODEL_TYPE / MODEL_ALPHA / N_LAGS
                  # experiments 61+:   free search

_keys = [k.strip() for k in OPENROUTER_API_KEYS.split(",") if k.strip()]
assert _keys, "Set at least one OpenRouter API key above"
print(f"Config OK: model={LLM_MODEL}, tag={RUN_TAG}, keys={len(_keys)}")


---
## Cell 1: Setup

In [ ]:
%%time
!pip install -q pandas pyarrow fastparquet scikit-learn xgboost lightgbm statsmodels matplotlib requests

import os
REQUIRED_FILES = ['prepare_data.py', 'baselines.py', 'nowcast.py', 'program_nowcast.md']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
if missing:
    print(f"Missing: {missing} -- upload them:")
    from google.colab import files
    files.upload()

if not os.path.exists('.git'):
    !git init && git config user.email "nowcastmy@colab" && git config user.name "NowcastMY"
    !git add -A && git commit -m "initial commit"

!python prepare_data.py --cache-dir {DATA_DIR}

import pandas as pd
panel = pd.read_parquet(f'{DATA_DIR}/panel_quarterly.parquet')
feature_cols = sorted([c for c in panel.columns if c not in ['date', 'gdp_real', 'gdp_real_sa'] and not c.startswith('gdp_')])
print(f"\nAvailable feature columns ({len(feature_cols)}):")
for c in feature_cols:
    n = panel[c].notna().sum()
    print(f"  {c:<40} {n:>3}/{len(panel)} valid")


---
## Cell 2: Baselines

In [ ]:
!python baselines.py --data-dir {DATA_DIR}

import pandas as pd
baselines_df = pd.read_csv(f"{DATA_DIR}/baselines_results.csv")
ar1_row = baselines_df[(baselines_df['model'] == 'AR(1)') & (baselines_df['variant'] == 'full')]
if len(ar1_row) > 0:
    BASELINE_AR1_RMSE = ar1_row.iloc[0]['rmse']
    print(f"\n>>> AR(1) RMSE to beat: {BASELINE_AR1_RMSE:.4f}")
else:
    print("WARNING: No AR(1) baseline found.")


---
## Cell 3: Agent Helpers

In [ ]:
import re, json, time, subprocess, requests
from datetime import datetime
from pathlib import Path
import pandas as pd, numpy as np

# -- API key rotation -------------------------------------------------------
_api_keys = [k.strip() for k in OPENROUTER_API_KEYS.split(",") if k.strip()]
_key_idx = 0

def _next_key():
    global _key_idx
    key = _api_keys[_key_idx % len(_api_keys)]
    _key_idx += 1
    return key

def call_llm(prompt, temperature=0.7, max_retries=3):
    for attempt in range(max_retries):
        key = _next_key()
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                json={"model": LLM_MODEL,
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 4096, "temperature": temperature},
                timeout=120)
            resp.raise_for_status()
            content = resp.json()["choices"][0]["message"]["content"]
            return content if content and len(content.strip()) > 10 else ""
        except Exception as e:
            print(f"  LLM error (attempt {attempt+1}, key ...{key[-6:]}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** (attempt + 1))
    return ""


# -- Diff parsing -----------------------------------------------------------
def parse_diff(response):
    """Robust parser for DESCRIPTION/OLD/NEW."""
    response = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL)
    description = old_code = new_code = None

    for pat in [r"DESCRIPTION[:\s]*\n(.*?)(?=\nOLD\b)",
                r"DESCRIPTION[:\s]+(.*?)(?=\nOLD\b)",
                r"DESCRIPTION[:\s]*\n(.*?)(?=\n```)",]:
        m = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if m: description = m.group(1).strip(); break

    for pat in [r"OLD[:\s]*\n```[^\n]*\n(.*?)```\s*(?=\nNEW\b)",
                r"OLD[:\s]*\n```[^\n]*\n(.*?)```",
                r"OLD[:\s]*\n(.*?)(?=\nNEW\b)",]:
        m = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if m: old_code = m.group(1).rstrip("\n"); break

    for pat in [r"NEW[:\s]*\n```[^\n]*\n(.*?)```", r"NEW[:\s]*\n(.*?)$"]:
        m = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if m: new_code = m.group(1).rstrip("\n"); break

    if not all([description, old_code, new_code]): return None
    old_code = old_code.strip("\n"); new_code = new_code.strip("\n")
    if len(old_code.strip()) < 5: return None
    return description, old_code, new_code


def apply_diff(filepath, old_code, new_code):
    try:
        content = open(filepath).read()
        if old_code in content:
            content = content.replace(old_code, new_code, 1)
        elif old_code.strip() in content:
            content = content.replace(old_code.strip(), new_code.strip(), 1)
        else:
            print(f"  OLD not found. Preview: {old_code[:80]}...")
            return False
        open(filepath, "w").write(content)
        return True
    except Exception as e:
        print(f"  Diff error: {e}"); return False


# -- Validation -------------------------------------------------------------
def validate_nowcast():
    """Check nowcast.py is not corrupted."""
    try:
        code = open("nowcast.py").read()
        if "def main()" not in code:
            print("  CORRUPT: main() missing"); return False
        if "RMSE_VS_AR1=" not in code:
            print("  CORRUPT: RMSE_VS_AR1 output missing"); return False
        compile(code, "nowcast.py", "exec")
        return True
    except SyntaxError as e:
        print(f"  CORRUPT: SyntaxError: {e}"); return False


# -- Git helpers ------------------------------------------------------------
def git_commit(msg):
    subprocess.run(["git", "add", "-A"], capture_output=True)
    subprocess.run(["git", "commit", "-am", msg], capture_output=True)

def git_tag_best():
    """Tag HEAD as the best-performing commit."""
    subprocess.run(["git", "tag", "-f", "best"], capture_output=True)

def git_revert():
    """Revert nowcast.py to the best tag if it exists, else HEAD."""
    r = subprocess.run(["git", "rev-parse", "--verify", "best"], capture_output=True)
    if r.returncode == 0:
        subprocess.run(["git", "checkout", "best", "--", "nowcast.py"], capture_output=True)
    else:
        subprocess.run(["git", "checkout", "--", "nowcast.py"], capture_output=True)


# -- Evaluation -------------------------------------------------------------
def run_nowcast():
    result = subprocess.run(
        ["python", "nowcast.py", "--data-dir", DATA_DIR],
        capture_output=True, text=True, timeout=120)
    stdout, stderr = result.stdout, result.stderr
    rmse_m = re.search(r"RMSE_VS_AR1=([\d.]+)", stdout)
    cov_m  = re.search(r"COVERAGE_90PCT=([\d.nan]+)", stdout)
    rmse = float(rmse_m.group(1)) if rmse_m else 999.0
    cov  = float(cov_m.group(1)) if cov_m and cov_m.group(1) != "nan" else float("nan")
    return {"rmse_vs_ar1": rmse, "coverage": cov,
            "stdout": stdout, "stderr": stderr, "crashed": rmse >= 999.0}


# -- TSV logging & state recovery -------------------------------------------
TSV_PATH = Path(DATA_DIR) / "results.tsv"

def log_result(exp_id, desc, result, status):
    with open(TSV_PATH, "a") as f:
        f.write(f"{exp_id}\t{datetime.now().isoformat()}\t{desc}\t"
                f"{result['rmse_vs_ar1']:.6f}\t{result['coverage']}\t{status}\n")

def load_state_from_tsv():
    """
    Recover best_rmse, best_coverage, consecutive_failures, next_exp_id from TSV.
    Returns (best_rmse, best_coverage, consecutive_failures, next_exp_id).
    """
    if not TSV_PATH.exists():
        return 999.0, float("nan"), 0, 0
    df = pd.read_csv(TSV_PATH, sep="\t", header=None,
                     names=["exp_id", "ts", "desc", "rmse", "cov", "status"])
    df["rmse"] = pd.to_numeric(df["rmse"], errors="coerce")
    df["exp_id"] = pd.to_numeric(df["exp_id"], errors="coerce")
    df = df.dropna(subset=["rmse", "exp_id"])
    if len(df) == 0:
        return 999.0, float("nan"), 0, 0
    valid = df[df["rmse"] < 900]
    best_rmse = valid["rmse"].min() if len(valid) else 999.0
    best_cov = float("nan")
    if len(valid):
        best_row = valid.loc[valid["rmse"].idxmin()]
        try: best_cov = float(best_row["cov"])
        except: pass
    # Trailing consecutive failures
    statuses = df["status"].astype(str).str.strip().tolist()
    consec = 0
    for s in reversed(statuses):
        if s in ("crash", "discard"): consec += 1
        else: break
    next_exp_id = int(df["exp_id"].max()) + 1
    return best_rmse, best_cov, consec, next_exp_id


# -- History summary for prompt ---------------------------------------------
def get_history_summary():
    if not TSV_PATH.exists(): return "No experiments yet."
    df = pd.read_csv(TSV_PATH, sep="\t", header=None,
                     names=["exp_id", "ts", "desc", "rmse", "cov", "status"])
    df["rmse"] = pd.to_numeric(df["rmse"], errors="coerce")
    df["status"] = df["status"].astype(str).str.strip()
    kept = df[df["status"] == "keep"][["exp_id", "rmse", "desc"]].copy()
    kept_str = kept.to_string(index=False) if len(kept) else "  (none yet)"
    recent_fail = df[df["status"].isin(["crash", "discard"])].tail(8)
    fail_lines = [f"  exp {int(r.exp_id):>3}: [{r.status}] {r.desc}" for _, r in recent_fail.iterrows()]
    fail_str = "\n".join(fail_lines) if fail_lines else "  (none)"
    return f"KEPT (improvements so far):\n{kept_str}\n\nRECENT FAILURES:\n{fail_str}"


# -- Config block extractor -------------------------------------------------
def extract_config_block():
    """Pull just the CONFIGURATION section from nowcast.py."""
    code = open("nowcast.py").read()
    m = re.search(
        r"(# ={3,}.*?CONFIGURATION.*?# ={3,}.*?\n)(.*?)(?=\ndef |\nclass |\nif __name__)",
        code, re.DOTALL)
    if m:
        return m.group(0).strip()
    return "\n".join(code.splitlines()[:30])


# -- Phase logic ------------------------------------------------------------
def get_phase_instruction(exp_id):
    if exp_id <= PHASE1_END:
        return (f"PHASE 1 (exp {exp_id}/{PHASE1_END}): Change ONLY FEATURE_COLS. "
                "Do not touch MODEL_TYPE, MODEL_ALPHA, N_LAGS, COVID_WEIGHT, or other params.")
    elif exp_id <= PHASE2_END:
        return (f"PHASE 2 (exp {exp_id}/{PHASE2_END}): Change ONLY MODEL_TYPE, MODEL_ALPHA, or N_LAGS. "
                "Keep FEATURE_COLS from the best experiment so far.")
    else:
        return (f"PHASE 3 (exp {exp_id}/{MAX_EXPERIMENTS}): Free search -- change ANY single config param "
                "to squeeze further improvement from the best config found so far.")


# -- Temperature schedule ---------------------------------------------------
def get_temperature(exp_id, consecutive_failures):
    """Anneal 0.85->0.35 over run; boost up to +0.3 when stuck."""
    base = 0.85 - (exp_id / MAX_EXPERIMENTS) * 0.5
    escape = min(0.3, consecutive_failures * 0.05)
    return round(min(0.95, max(0.25, base + escape)), 2)


# -- Prompt builder ---------------------------------------------------------
_panel = pd.read_parquet(f'{DATA_DIR}/panel_quarterly.parquet')
_all_cols = sorted([c for c in _panel.columns
                    if c not in ["date", "gdp_real", "gdp_real_sa"] and not c.startswith("gdp_")])
_col_list = ", ".join(_all_cols[:40])
del _panel

def build_prompt(exp_id, best_rmse, best_coverage, consecutive_failures):
    config_block = extract_config_block()
    history = get_history_summary()
    phase_instr = get_phase_instruction(exp_id)
    cov_str = f"{best_coverage:.3f}" if not np.isnan(best_coverage) else "unknown"
    return (
        "You are an autonomous research agent improving a GDP nowcasting model for Malaysia.\n\n"
        "## Current status\n"
        f"- Best RMSE_VS_AR1 so far: {best_rmse:.4f}  (target: < 1.0, stretch: < 0.80)\n"
        f"- Best COVERAGE_90PCT so far: {cov_str}  (target: > 0.85)\n"
        f"- Consecutive failures: {consecutive_failures}\n\n"
        "## Phase instruction\n"
        f"{phase_instr}\n\n"
        "## Available feature columns (EXACT names -- use only these in FEATURE_COLS)\n"
        f"{_col_list}\n\n"
        "## Current configuration block\n"
        "```python\n"
        f"{config_block}\n"
        "```\n\n"
        "## Experiment history\n"
        f"{history}\n\n"
        "## Rules\n"
        "- Only change the CONFIGURATION section at the top of nowcast.py\n"
        "- Allowed params: FEATURE_COLS, MODEL_TYPE, MODEL_ALPHA, N_LAGS, USE_SURPRISE, COVID_WEIGHT, BOOTSTRAP_N, CONFIDENCE_LEVEL\n"
        "- Do NOT modify any function definitions (build_features, get_model, expanding_window_eval, ar1_baseline, main)\n"
        "- FEATURE_COLS must only use names from the available columns list above\n"
        "- COVID_WEIGHT: 0.0 = exclude COVID quarters, 1.0 = include fully, 0.3 = downweight\n"
        "- Max 15 features. Change exactly ONE param per experiment.\n\n"
        f"## Your task (experiment #{exp_id})\n"
        "Propose ONE change. Respond in EXACTLY this format with no other text:\n\n"
        "DESCRIPTION\n"
        "one line explaining the change\n\n"
        "OLD\n"
        "```python\n"
        "exact lines to replace (must match nowcast.py exactly)\n"
        "```\n\n"
        "NEW\n"
        "```python\n"
        "replacement lines\n"
        "```\n"
    )

print(f"Agent helpers loaded. Model: {LLM_MODEL}, Keys: {len(_api_keys)}")
print(f"Features available ({len(_all_cols)}): {_col_list[:200]}...")


---
## Cell 4: Agent Loop
Interrupt the kernel to stop. Re-run this cell to resume -- state is recovered from TSV.

In [ ]:
import numpy as np

!git checkout -b nowcast/{RUN_TAG} 2>/dev/null || git checkout nowcast/{RUN_TAG}

# -- Recover state from TSV ------------------------------------------------
best_rmse, best_coverage, consecutive_failures, next_exp_id = load_state_from_tsv()

if next_exp_id == 0:
    # Fresh run -- establish baseline
    print("Running baseline (experiment 0)...")
    baseline_result = run_nowcast()
    best_rmse = baseline_result["rmse_vs_ar1"]
    best_coverage = baseline_result["coverage"]
    if best_rmse >= 999.0:
        print(f"  Baseline CRASHED. stderr: {baseline_result[\"stderr\"][-300:]}")
        print("  Fix nowcast.py before continuing.")
    else:
        print(f"  Baseline RMSE_VS_AR1 = {best_rmse:.4f}")
        print(f"  Baseline COVERAGE    = {best_coverage}")
        git_commit("experiment 0: baseline")
        git_tag_best()
        log_result(0, "baseline", baseline_result, "keep")
    next_exp_id = 1
else:
    print(f"Resuming from experiment {next_exp_id}.")
    print(f"  Recovered best_rmse = {best_rmse:.4f}, consecutive_failures = {consecutive_failures}")

# -- Main loop --------------------------------------------------------------
for exp_id in range(next_exp_id, MAX_EXPERIMENTS + 1):
    print(f"\n{'='*60}")
    print(f"  Experiment {exp_id}/{MAX_EXPERIMENTS}  |  Best: {best_rmse:.4f}  |  Consec. fail: {consecutive_failures}")
    print(f"{'='*60}")

    temp = get_temperature(exp_id, consecutive_failures)
    prompt = build_prompt(exp_id, best_rmse, best_coverage, consecutive_failures)

    print(f"  Calling LLM (temp={temp})...", end=" ", flush=True)
    response = call_llm(prompt, temperature=temp)
    if not response:
        print("EMPTY")
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping."); break
        continue
    print("OK")

    parsed = parse_diff(response)
    if parsed is None:
        print(f"  Parse failed. Preview: {response[:120]}...")
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping."); break
        continue

    description, old_code, new_code = parsed
    print(f"  Desc: {description[:80]}")

    if not apply_diff("nowcast.py", old_code, new_code):
        git_revert()
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping."); break
        continue

    if not validate_nowcast():
        git_revert()
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping."); break
        continue

    git_commit(f"experiment {exp_id}: {description[:60]}")

    print("  Running...", end=" ", flush=True)
    try:
        result = run_nowcast()
    except Exception as e:
        print(f"ERROR: {e}")
        result = {"rmse_vs_ar1": 999.0, "coverage": float("nan"),
                  "stdout": "", "stderr": str(e), "crashed": True}

    if result["crashed"]:
        print("CRASHED")
        if result["stderr"]: print(f"  {result[\"stderr\"][-200:]}")
        git_revert()
        git_commit(f"revert {exp_id} (crash)")
        log_result(exp_id, description, result, "crash")
        consecutive_failures += 1

    elif result["rmse_vs_ar1"] < best_rmse:
        print(f"IMPROVED! {result[\"rmse_vs_ar1\"]:.4f} (was {best_rmse:.4f})")
        best_rmse = result["rmse_vs_ar1"]
        best_coverage = result["coverage"]
        consecutive_failures = 0
        git_tag_best()
        log_result(exp_id, description, result, "keep")

    else:
        print(f"No improvement: {result[\"rmse_vs_ar1\"]:.4f} >= {best_rmse:.4f}")
        git_revert()
        git_commit(f"revert {exp_id}")
        consecutive_failures = 0
        log_result(exp_id, description, result, "discard")

    if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
        print(f"\n  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping."); break

    time.sleep(1)

print(f"\n{'='*60}")
print(f"  Done. Best RMSE_VS_AR1: {best_rmse:.4f}")
print(f"{'='*60}")


---
## Cell 5: Analysis

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

tsv = pd.read_csv(f'{DATA_DIR}/results.tsv', sep='\t', header=None,
                  names=['exp_id','timestamp','description','rmse_vs_ar1','coverage','status'])
tsv['rmse_vs_ar1'] = pd.to_numeric(tsv['rmse_vs_ar1'], errors='coerce')
tsv['exp_id'] = pd.to_numeric(tsv['exp_id'], errors='coerce')
tsv = tsv.dropna(subset=['rmse_vs_ar1', 'exp_id'])
tsv['exp_id'] = tsv['exp_id'].astype(int)
tsv_valid = tsv[tsv['rmse_vs_ar1'] < 900].copy().reset_index(drop=True)
tsv['status'] = tsv['status'].astype(str).str.strip()
tsv_valid['status'] = tsv_valid['status'].astype(str).str.strip()

n_keep  = (tsv['status'] == 'keep').sum()
n_disc  = (tsv['status'] == 'discard').sum()
n_crash = (tsv['status'] == 'crash').sum()
print(f'Total: {len(tsv)} | Kept: {n_keep} | Discarded: {n_disc} | Crashed: {n_crash}')

if len(tsv_valid) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Panel 1 -- RMSE trajectory
    ax = axes[0]
    x_all = tsv_valid['exp_id'].values
    y_all = tsv_valid['rmse_vs_ar1'].values
    best_line = np.minimum.accumulate(y_all)

    for _, row in tsv_valid.iterrows():
        color = '#2E75B6' if row['status'] == 'keep' else '#CCC' if row['status'] == 'discard' else '#E24B4A'
        ax.scatter(row['exp_id'], row['rmse_vs_ar1'], c=color, s=30, zorder=3)

    ax.plot(x_all, best_line, 'r-', lw=2, label=f'Best (final: {best_line[-1]:.4f})', zorder=4)
    ax.axhline(1.0, color='gray', ls='--', lw=0.8, label='AR(1) baseline')
    ax.axhline(0.8, color='green', ls=':', lw=0.8, label='Stretch target (0.80)')
    for phase_end in [PHASE1_END, PHASE2_END]:
        ax.axvline(phase_end, color='orange', ls='--', lw=0.8, alpha=0.6)
    ax.set_ylabel('RMSE / AR(1)')
    ax.set_title(f'NowcastMY v4 -- Best RMSE ratio: {best_line[-1]:.4f}')
    ax.legend(fontsize=9)

    # Panel 2 -- Coverage
    ax2 = axes[1]
    cov_valid = tsv_valid.dropna(subset=['coverage'])
    if len(cov_valid) > 0:
        cov_vals = pd.to_numeric(cov_valid['coverage'], errors='coerce')
        ax2.scatter(cov_valid['exp_id'], cov_vals, s=20, color='#5B9BD5', alpha=0.6, label='Coverage')
        ax2.axhline(0.90, color='green', ls=':', lw=0.8, label='Nominal 90%')
        ax2.axhline(0.85, color='orange', ls=':', lw=0.8, label='Floor (0.85)')
        ax2.set_ylabel('Coverage (90% PI)')
        ax2.set_ylim(0, 1.05)
        ax2.legend(fontsize=9)
    ax2.set_xlabel('Experiment')

    plt.tight_layout()
    plt.savefig(f'{DATA_DIR}/results.png', dpi=150)
    plt.show()

    best_idx = np.argmin(y_all)
    best_row = tsv_valid.iloc[best_idx]
    print(f"\nBest: experiment {int(best_row['exp_id'])}, RMSE_VS_AR1 = {best_row['rmse_vs_ar1']:.4f}")
    print(f"Description: {best_row['description']}")
    if best_row['rmse_vs_ar1'] < 0.80:
        print('>>> PRD TARGET MET! <<<')
else:
    print('No valid experiments to plot.')


---
## Cell 6: Export

In [ ]:
from google.colab import files
!cp nowcast.py {DATA_DIR}/best_nowcast.py 2>/dev/null
!tar czf nowcastmy_results.tar.gz {DATA_DIR}/results.tsv {DATA_DIR}/best_nowcast.py {DATA_DIR}/results.png 2>/dev/null
files.download("nowcastmy_results.tar.gz")
